In [1]:
import torch
from transformers import MarianMTModel, MarianTokenizer

device = torch.device("cpu")

translator_model = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(translator_model)
translator = MarianMTModel.from_pretrained(translator_model)

examples = [
    "I love cats and dogs",
    "Hello, how are you?",
    "The weather is nice today"
]

for example in examples:
    tokens = tokenizer(example, return_tensors="pt").to(device)
    translated = translator.generate(**tokens)
    result = tokenizer.decode(translated[0], skip_special_tokens=True)
    print(result)

/home/s2883781/.conda/envs/mls/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Я люблю кошек и собак.
Привет, как дела?
Сегодня хорошая погода.


## Seq2Seq translation model

In [2]:
from datasets import load_dataset
ds = load_dataset("Helsinki-NLP/opus-100", "en-ru")
print(ds["train"][0])

{'translation': {'en': "Yeah, that's not exactly...", 'ru': 'Да, но не совсем...'}}


In [3]:
print(ds)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [4]:
train_ds = ds['train'].select(range(50000))

In [5]:
import string
def preprocess(sentence):
  sentence = sentence.lower()
  cleaner = str.maketrans('', '', string.punctuation)
  sentence = sentence.translate(cleaner)
  sentence = sentence.split(' ')
  tokens = ["<SOS>"] + sentence + ["<EOS>"]

  return tokens



In [6]:
print(preprocess("Hello, I love my cat - Mushi very much!"))

['<SOS>', 'hello', 'i', 'love', 'my', 'cat', '', 'mushi', 'very', 'much', '<EOS>']


In [7]:
eng_sentences = []
ru_sentences = []
for line in train_ds:
  eng_sentences.append(preprocess(line['translation']['en']))
  ru_sentences.append(preprocess(line['translation']['ru']))

In [8]:
class Vocabulary:
  def __init__(self):
    self.word2index = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    self.index2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
    self.n_words = 4

  def add_sentence(self, sentence):
    for word in sentence:
      self.add_word(word)

  def add_word(self, word):
    if word not in self.word2index:
      self.word2index[word] = self.n_words
      self.index2word[self.n_words] = word
      self.n_words += 1

In [9]:
eng_vocab = Vocabulary()
for s in eng_sentences:
  eng_vocab.add_sentence(s)
ru_vocab = Vocabulary()
for s in ru_sentences:
  ru_vocab.add_sentence(s)

In [10]:
print(eng_vocab.n_words)
print(ru_vocab.n_words)
# russian is morphologicaly richer than english

40285
76395


In [11]:
def sentence_to_indices(sentence, vocab):
  indices = []
  for word in sentence:
    word = vocab.word2index[word]
    indices.append(word)
  return indices

print(sentence_to_indices(eng_sentences[0], eng_vocab))

[1, 4, 5, 6, 7, 2]


In [12]:
eng_indices = []
ru_indices = []
for eng_sentence in eng_sentences:
  eng_indices.append(sentence_to_indices(eng_sentence,eng_vocab))
for ru_sentence in ru_sentences:
  ru_indices.append(sentence_to_indices(ru_sentence,ru_vocab))

In [13]:
print(eng_sentences[11])
print(ru_sentences[11])

['<SOS>', 'you', 'told', 'dan', 'i', 'was', 'here', '<EOS>']
['<SOS>', 'ты', 'сказал', 'дену', 'что', 'я', 'здесь', 'был', '<EOS>']


In [14]:
class TranslationDataset:
  def __init__(self,eng_indices,ru_indices):
    self.eng_ind = eng_indices
    self.ru_ind = ru_indices

  def __len__(self):
    return len(self.eng_ind)

  def __getitem__(self,i):
    return self.eng_ind[i], self.ru_ind[i]

In [15]:
from torch.nn.utils.rnn import pad_sequence
def collate_fn(batch):
  eng_batch, ru_batch = zip(*batch)
  eng_batch = [torch.tensor(s) for s in eng_batch]
  ru_batch = [torch.tensor(s) for s in ru_batch]
  eng_batch = pad_sequence(eng_batch, batch_first=True, padding_value=0)
  ru_batch = pad_sequence(ru_batch, batch_first=True,  padding_value=0)
  return eng_batch, ru_batch

In [16]:
from torch.utils.data import DataLoader
dataset = TranslationDataset(eng_indices, ru_indices)
loader = DataLoader(dataset, 32, collate_fn=collate_fn)

In [17]:
for eng, ru in loader:
    print(eng.shape)
    print(ru.shape)
    break

torch.Size([32, 49])
torch.Size([32, 34])


### Encoder - Decoder

In [18]:
import torch
import torch.nn as nn
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, dropout):
    super().__init__()
    self.vocab_size = vocab_size
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, n_layers, dropout=dropout)

  def forward(self, x):
    x = self.embedding(x)
    output, hidden = self.rnn(x)
    return hidden

In [21]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, dropout):
    super().__init__()
    self.vocab_size = vocab_size
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, n_layers, dropout=dropout)
    self.fc = nn.Linear(hidden_dim, vocab_size)
  
  def forward(self,x, hidden):
    x = self.embedding(x)
    output, hidden = self.rnn(x, hidden)
    prediction = self.fc(output)
    return prediction, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        hidden = self.encoder(src)
        input = torch.ones(trg.shape[0], dtype=torch.long)
        for i in range(trg.shape[1]):
            prediction, hidden = self.decoder(input, hidden)
            